# ⚡ Open Generative AI — 100% Free Kaggle GPU Studio
### Free Kaggle Dual T4 GPUs • 30 Hours/Week Free • Zero Filters

👉 **Step 1:** Right Sidebar me **Settings → Accelerator → GPU T4 x2** select karein.
👉 **Step 2:** **Internet: ON** toggle karein.
👉 **Step 3:** Run All cells (Shift + Enter).

In [ ]:
# 1. Install required packages
!pip install -q diffusers transformers accelerate torch torchvision torchaudio gradio pyngrok

In [ ]:
# 2. Launch Uncensored Cloud GPU Studio
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import gradio as gr

print("[*] Loading model on Kaggle GPU...")
model_id = "Lykon/DreamShaper"
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to("cuda")
pipe.enable_attention_slicing()
print("[*] Ready on Kaggle GPU!")

def generate_image(prompt, negative_prompt, steps, guidance_scale, width, height, seed):
    generator = None
    if seed != -1:
        generator = torch.Generator("cuda").manual_seed(int(seed))
    else:
        seed = int(torch.randint(0, 2147483647, (1,)).item())
        generator = torch.Generator("cuda").manual_seed(seed)
        
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=int(steps),
        guidance_scale=float(guidance_scale),
        width=int(width),
        height=int(height),
        generator=generator
    ).images[0]
    
    return image, f"Generated on Kaggle GPU with Seed: {seed}"

with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald"), title="Open Generative AI Kaggle GPU") as demo:
    gr.Markdown("# ⚡ Open Generative AI — Kaggle Cloud GPU")
    gr.Markdown("**30 Hours Free / Week • Dual NVIDIA T4 • Zero Content Filter**")
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Prompt", placeholder="Describe what you want to generate...", lines=3, value="a stunning realistic portrait of a beautiful woman, 8k uhd, masterpiece, cinematic lighting")
            negative_prompt = gr.Textbox(label="Negative Prompt", lines=2, value="ugly, blurry, deformed hands, bad anatomy, low quality, cartoon")
            
            with gr.Row():
                steps = gr.Slider(minimum=10, maximum=50, value=25, step=1, label="Sampling Steps")
                guidance_scale = gr.Slider(minimum=1.0, maximum=15.0, value=7.5, step=0.5, label="CFG Scale")
                
            with gr.Row():
                width = gr.Dropdown(choices=[512, 768], value=512, label="Width")
                height = gr.Dropdown(choices=[512, 768], value=512, label="Height")
                seed = gr.Number(value=-1, label="Seed (-1 for random)")
                
            generate_btn = gr.Button("🚀 Generate on Kaggle GPU", variant="primary")
            info_text = gr.Markdown("")
            
        with gr.Column(scale=1):
            output_image = gr.Image(label="Generated Image", type="pil")
            
    generate_btn.click(
        fn=generate_image,
        inputs=[prompt, negative_prompt, steps, guidance_scale, width, height, seed],
        outputs=[output_image, info_text]
    )

demo.launch(share=True, debug=True)